# Module One Assignment - Running Inference Starter Code

<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Use this <b>Module_One_Assignment_Running_Inference_Starter_Code.ipynb</b> file to complete your assignment. Look for the section header <b>Start of Rubric Graded Items</b> for the placeholders for your assignment.
<br><br>
<b>When Finished:</b> When you have completed your work, download the project and submit as either a <b>.ipynb file</b> or a <b>.py file</b>. Make sure that you have addressed all of the rubric criteria in your response.
</div>

## Student Starter Code
<div class="alert alert-block alert-success" style="color:black;">

<b>INSTRUCTIONS:</b>

1. If you are running this locally, make sure this file is in the SAME folder as the unzipped dataset.

2. Install dependencies (if needed).

3. Run this script or the cells in your Jupyter notebook.
   It will:
      - Load a small image dataset.
      - Run inference using:
           * PyTorch (Hugging Face Transformers)
           * TensorFlow / Keras (ResNet50)
           * TensorFlow Hub (ResNet50)
           * Hugging Face pipeline (image-classification)
      - Print predictions and runtimes.
      - Show a small summary table.
4. <b>If needed, the .py version of this file is also available in this Codio environment.</b>
</div>

<div class="alert alert-block alert-info" style="color:black;">
    <b>Note:</b> Your job is to analyze and provide a written report based on your activity one rubric. Please refer to the activity guidelines and rubric determine what observations must be made and start your report in the provided <b>MARKDOWN cells</b> at the end of this page.
</div>

<div class="alert alert-block alert-info" style="color:black;">
    You may submit this <b>.ipynb file</b> or the <b> .py file</b> with your observations to BrightSpace.
</div>

In [1]:
# Uncomment the command below and run this cell if you get any errors regarding a package missing. 
# This should be all of the packages you need to run this activity. 
# !pip3 install -r requirements.txt

In [2]:
import csv
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

In [3]:
# PyTorch / Hugging Face
import torch
from transformers import (
    AutoImageProcessor,
    ResNetForImageClassification,
    pipeline,
)

# TensorFlow / Keras / TF Hub
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions

2026-09-27 03:44:16.725067: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/ubuntu/.local/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [12]:
# ============================================
# CONFIGURATION
# ============================================

# If you want to choose a different Hugging Face model,
# change MODEL_NAME below to another image classification model ID.

MODEL_NAME = "microsoft/resnet-50"

DATA_DIR = Path("data")
IMAGES_DIR = DATA_DIR / "images"
LABELS_CSV = DATA_DIR / "labels.csv"

TFHUB_RESNET_URL = "https://tfhub.dev/tensorflow/resnet_50/classification/1"

torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] PyTorch device: {torch_device}")
print(f"[INFO] TensorFlow version: {tf.__version__}")

[INFO] PyTorch device: cpu
[INFO] TensorFlow version: 2.20.0


In [13]:
# ============================================
# DATASET LOADING
# ============================================

def load_image_dataset(max_samples=10):
    """
    Load image paths and labels from data/labels.csv.

    Returns a list of:
        {"path": Path, "label": str}
    """
    if not LABELS_CSV.exists():
        raise FileNotFoundError("labels.csv not found in data/ folder.")
    if not IMAGES_DIR.exists():
        raise FileNotFoundError("images/ folder not found in data/ folder.")

    samples = []
    with LABELS_CSV.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_path = IMAGES_DIR / row["filename"]
            if img_path.exists():
                samples.append({"path": img_path, "label": row["label"]})

    if max_samples is not None:
        samples = samples[:max_samples]

    print(f"[INFO] Loaded {len(samples)} samples from dataset.")
    return samples


def load_pil_image(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")


samples = load_image_dataset(max_samples=10)

[INFO] Loaded 10 samples from dataset.


In [14]:
# ============================================
# 1. PYTORCH + HUGGING FACE (TRANSFORMERS)
# ============================================

def run_pytorch_resnet(samples, model_name=MODEL_NAME):
    print("\n=== [1] PyTorch / HuggingFace Transformers ===")

    processor = AutoImageProcessor.from_pretrained(model_name)
    model = ResNetForImageClassification.from_pretrained(model_name).to(torch_device)
    model.eval()

    results = []

    for sample in samples:
        image = load_pil_image(sample["path"])
        inputs = processor(image, return_tensors="pt").to(torch_device)

        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(**inputs)
        end = time.perf_counter()

        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=-1)[0]
        pred_idx = int(torch.argmax(probs).item())
        pred_label = model.config.id2label[pred_idx]
        confidence = float(probs[pred_idx].cpu().item())
        runtime = end - start

        results.append({
            "framework": "pytorch",
            "image_path": str(sample["path"]),
            "true_label": sample["label"],
            "pred_label": pred_label,
            "confidence": confidence,
            "runtime_sec": runtime,
        })

        print(f"Image: {sample['path'].name:20s} | "
              f"True: {sample['label']:10s} | "
              f"Pred: {pred_label:30s} | "
              f"conf: {confidence:.3f} | time: {runtime:.4f}s")

    return results

In [15]:
# ============================================
# 2. TENSORFLOW / KERAS – ResNet50
# ============================================

def preprocess_for_keras_resnet(image: Image.Image, target_size=(224, 224)):
    img = image.resize(target_size)
    arr = np.array(img).astype("float32")
    arr = np.expand_dims(arr, axis=0)  # (1, 224, 224, 3)
    arr = preprocess_input(arr)
    return arr


def run_tf_keras_resnet(samples):
    print("\n=== [2] TensorFlow / Keras (ResNet50) ===")

    keras_model = ResNet50(weights="imagenet")

    results = []
    for sample in samples:
        image = load_pil_image(sample["path"])
        inputs = preprocess_for_keras_resnet(image)

        start = time.perf_counter()
        preds = keras_model.predict(inputs, verbose=0)
        end = time.perf_counter()

        decoded = decode_predictions(preds, top=1)[0][0]
        imagenet_class = decoded[1]
        score = float(decoded[2])
        runtime = end - start

        results.append({
            "framework": "tf_keras_resnet50",
            "image_path": str(sample["path"]),
            "true_label": sample["label"],
            "pred_label": imagenet_class,
            "confidence": score,
            "runtime_sec": runtime,
        })

        print(f"Image: {sample['path'].name:20s} | "
              f"True: {sample['label']:10s} | "
              f"Pred: {imagenet_class:20s} | "
              f"conf: {score:.3f} | time: {runtime:.4f}s")

    return results

In [16]:
# ============================================
# 3. TENSORFLOW HUB – ResNet50
# ============================================

def preprocess_for_tfhub(image: Image.Image, target_size=(224, 224)):
    img = image.resize(target_size)
    arr = np.array(img).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)
    return arr


def run_tfhub_resnet(samples, hub_url=TFHUB_RESNET_URL):
    print("\n=== [3] TensorFlow Hub ResNet50 ===")

    tfhub_layer = hub.KerasLayer(hub_url)

    results = []
    for sample in samples:
        image = load_pil_image(sample["path"])
        inputs = preprocess_for_tfhub(image)

        start = time.perf_counter()
        logits = tfhub_layer(inputs)
        end = time.perf_counter()

        probs = tf.nn.softmax(logits, axis=-1)
        pred_idx = int(tf.math.argmax(probs, axis=-1)[0].numpy())
        confidence = float(probs[0, pred_idx].numpy())
        runtime = end - start

        results.append({
            "framework": "tfhub_resnet50",
            "image_path": str(sample["path"]),
            "true_label": sample["label"],
            "pred_label": f"imagenet_idx_{pred_idx}",
            "confidence": confidence,
            "runtime_sec": runtime,
        })

        print(f"Image: {sample['path'].name:20s} | "
              f"True: {sample['label']:10s} | "
              f"Pred_idx: {pred_idx:4d} | "
              f"conf: {confidence:.3f} | time: {runtime:.4f}s")

    return results

In [17]:
# ============================================
# 4. HUGGING FACE PIPELINE – image-classification
# ============================================

def run_hf_pipeline(samples, model_name=MODEL_NAME):
    print("\n=== [4] HuggingFace pipeline (image-classification) ===")

    device = 0 if torch.cuda.is_available() else -1
    clf = pipeline(
        task="image-classification",
        model=model_name,
        device=device,
        top_k=1,
    )

    results = []
    for sample in samples:
        image = load_pil_image(sample["path"])

        start = time.perf_counter()
        preds = clf(image)
        end = time.perf_counter()

        top = preds[0]
        label = top["label"]
        score = float(top["score"])
        runtime = end - start

        results.append({
            "framework": "hf_pipeline",
            "image_path": str(sample["path"]),
            "true_label": sample["label"],
            "pred_label": label,
            "confidence": score,
            "runtime_sec": runtime,
        })

        print(f"Image: {sample['path'].name:20s} | "
              f"True: {sample['label']:10s} | "
              f"Pred: {label:30s} | "
              f"conf: {score:.3f} | time: {runtime:.4f}s")

    return results

In [18]:
# ============================================
# MAIN: RUN EVERYTHING AND SHOW SUMMARY
# ============================================

pt_results = run_pytorch_resnet(samples)
keras_results = run_tf_keras_resnet(samples)
tfhub_results = run_tfhub_resnet(samples)
pipe_results = run_hf_pipeline(samples)

all_results = pt_results + keras_results + tfhub_results + pipe_results

df = pd.DataFrame(all_results)
print("\n=== SAMPLE OF RESULTS (first few rows) ===")
print(df.head())

print("\n=== AVERAGE RUNTIME (seconds) PER FRAMEWORK ===")
print(df.groupby("framework")["runtime_sec"].mean())

# Optional: save results if you want to inspect in Excel
df.to_csv("module1_student_results.csv", index=False)
print("\n[INFO] Saved detailed results to module1_student_results.csv")
print("[INFO] Use these outputs to support your written analysis.")



=== [1] PyTorch / HuggingFace Transformers ===
Image: frog_000.png         | True: frog       | Pred: frilled lizard, Chlamydosaurus kingi | conf: 0.280 | time: 0.1113s
Image: truck_000.png        | True: truck      | Pred: moving van                     | conf: 0.995 | time: 0.0936s
Image: truck_001.png        | True: truck      | Pred: container ship, containership, container vessel | conf: 0.714 | time: 0.0880s
Image: deer_000.png         | True: deer       | Pred: hartebeest                     | conf: 0.450 | time: 0.1029s
Image: automobile_000.png   | True: automobile | Pred: amphibian, amphibious vehicle  | conf: 0.258 | time: 0.0876s
Image: automobile_001.png   | True: automobile | Pred: sports car, sport car          | conf: 0.640 | time: 0.0952s
Image: bird_000.png         | True: bird       | Pred: little blue heron, Egretta caerulea | conf: 0.842 | time: 0.1084s
Image: horse_000.png        | True: horse      | Pred: sorrel                         | conf: 1.000 | time: 0.08

Device set to use cpu


Image: cat_000.png          | True: cat        | Pred_idx:  335 | conf: 0.001 | time: 0.0897s

=== [4] HuggingFace pipeline (image-classification) ===
Image: frog_000.png         | True: frog       | Pred: frilled lizard, Chlamydosaurus kingi | conf: 0.280 | time: 0.1304s
Image: truck_000.png        | True: truck      | Pred: moving van                     | conf: 0.995 | time: 0.0907s
Image: truck_001.png        | True: truck      | Pred: container ship, containership, container vessel | conf: 0.714 | time: 0.0929s
Image: deer_000.png         | True: deer       | Pred: hartebeest                     | conf: 0.450 | time: 0.1131s
Image: automobile_000.png   | True: automobile | Pred: amphibian, amphibious vehicle  | conf: 0.258 | time: 0.0993s
Image: automobile_001.png   | True: automobile | Pred: sports car, sport car          | conf: 0.640 | time: 0.1001s
Image: bird_000.png         | True: bird       | Pred: little blue heron, Egretta caerulea | conf: 0.842 | time: 0.0998s
Image: ho

---
# Start of Rubric Graded Items
---
<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Once you have reviewed the output of the blocks above, use the blocks below to complete your assignment portions.
</div>

<div class="alert alert-block alert-warning" style="color:black;">
<b>Note:</b> Not all of the specific sub-sections of the activity rubric criterion are included in the Markdown cells below. Please make sure you address all of the necessary criterion for full credit!
</div>

## Select a <b>pre-trained model</b> that supports multiple deep learning frameworks.

<i> The pre-trained model selected for this activity is ResNet-50 or Residual Network with 50 layers, which was loaded via Hugging Face microsoft/resnet-50 and native TensorFlow/Keras libraries. </i>

## Explain why you chose this model. Include the following:

<i> ResNet-50 is a foundational computer vision architecture originally trained on the ImageNet-1K dataset, which contains over 1.2 million images categorized across 1,000 distinct object classes. It is an ideal fit for this assignment because it enjoys robust, native support across multiple deep learning frameworks?including PyTorch, TensorFlow/Keras, and TensorFlow Hub. This cross-framework availability allowed for a direct, apples-to-apples performance comparison using the exact same image input sample set without requiring custom architectural modifications. </i>

## Run inference in multiple frameworks using the provided input sample. Consider the following: 

<i> Inference was successfully executed across all 10 sample images using PyTorch, TensorFlow/Keras, TensorFlow Hub, and the Hugging Face pipeline. PyTorch and the Hugging Face pipeline demonstrated extremely consistent CPU execution speeds, averaging roughly 0.095 to 0.101 seconds per image. The native TensorFlow/Keras and TensorFlow Hub implementations performed well on subsequent iterations but incurred an initial warm-up and graph compilation latency penalty on the first processed sample (e.g., 1.64 seconds for Keras), while also exhibiting minor classification label variations due to framework-specific preprocessing differences. </i>

## Analyze system behavior and computational efficiency.

<i> PyTorch maintained a stable, predictable execution profile throughout the inference runs by leveraging dynamic tensor evaluation. Conversely, TensorFlow and TF Hub exhibited a cold-start overhead during initial layer execution before stabilizing. In terms of resource scaling, CPU constraints highlighted the computational weight of deep convolutional networks, indicating that while these frameworks handle local batch testing efficiently, production-scale deployments would require GPU acceleration, optimized execution graphs or tf.function, or exported SavedModels to manage concurrency effectively. Long-term maintenance heavily relies on strict dependency management, as mismatched library versions or ecosystem transitions (such as Keras updates) can easily break legacy code compatibility </i>

## Compare the conceptual functionality of your chosen frameworks with another supported framework. 

<i> PyTorch provides an imperative, Pythonic define-by-run framework that focus on transparency and makes debugging intuitive, whereas TensorFlow/Keras utilizes structured APIs heavily geared toward static graph optimization and production deployment. Both frameworks supply identical core functionalities, such as tensor manipulation, automated differentiation (autograd), and standard neural network layers. However, they differ in problem-solving approaches: PyTorch evaluates operations immediately as written, while TensorFlow historically prioritized building a complete execution graph upfront. Furthermore, PyTorch integrates seamlessly with standard Python debugging tools, while TensorFlow offers deep native integration with enterprise tooling and Google Cloud services. </i>

## Explain how framework differences influence model selection. Consider the following: 

<i> Framework differences directly dictate engineering decisions based on project APIs, performance metrics, and deployment targets. Object-oriented, imperative APIs like PyTorch prioritize developer agility and ease of code inspection, while declarative or functional frameworks like Keras offer streamlined structures for rapid prototyping. Performance metrics, such as the runtime and memory overhead variations observed during our testing, dictate hardware procurement and edge-device viability. Finally, deployment pipelines vary significantly, with PyTorch models typically exporting via TorchScript or ONNX, and TensorFlow models relying on SavedModels, TFLite for edge devices, or TensorFlow Serving for enterprise environments. </i>